### Loads in cleaned data files from 00_pull and produces the specific measures needed to create figures/tables in 02_analysis

#### **Inputs**
data/southafrica_all.dta
data/brazil_lstatus.csv, data/brazil_socialsecurity.csv, data/brazil_income.csv, data/brazil_hours.csv

#### **Outputs**
data/southafrica_analysis.dta
data/brazil_socialsecurity_analysis.csv, data/brazil_income_analysis.csv, data/brazil_hours_analysis.csv

# 01_clean notebook

### Load in data (from 00_pull)

In [17]:
import pandas as pd
data =  "../data/" # same convention as in previous notebook
southafrica_df = pd.read_stata(data + "southafrica_all.dta") # these are all the files created in 00_pull
brazil_contrib = pd.read_csv(data + "brazil_socialsecurity.csv")
brazil_income = pd.read_csv(data + "brazil_income.csv")
brazil_hours = pd.read_csv(data + "brazil_hours.csv")
brazil_lstatus = pd.read_csv(data + "brazil_lstatus.csv")

# double check to make sure its all loaded
print(southafrica_df.head())
print(brazil_contrib.head())
print(brazil_income.head())
print(brazil_hours.head())
print(brazil_lstatus.head())

  metro_code                                sector2  hrswrk      weight  \
0  Non-Metro  Formal sector (Including agriculture)    80.0  338.900242   
1  Non-Metro                         Not applicable     NaN  338.900242   
2  Non-Metro  Formal sector (Including agriculture)    84.0  528.518228   
3  Non-Metro  Formal sector (Including agriculture)    84.0  528.518228   
4  Non-Metro                     Private households    40.0  228.240240   

       province q13gender  q14age q15population  \
0  Western Cape      Male      54         White   
1  Western Cape    Female      47         White   
2  Western Cape      Male      62         White   
3  Western Cape    Female      58         White   
4  Western Cape    Female      48      Coloured   

                         q17education  \
0          Grade 10/Standard 8/Form 3   
1  Grade 12/Standard 10/Form 5/Matric   
2        Diploma with Grade 12/Std 10   
3  Grade 12/Standard 10/Form 5/Matric   
4           Grade 8/Standard 6/Form 1

### South Africa

In [20]:
# metro names are inconsistent across the survey years, so need to standardize
# make all lower case, removing neccessary spaces 
southafrica_df["metro_clean"] = southafrica_df["metro_code"].astype(str).str.strip().str.lower()
print(southafrica_df["metro_clean"].unique()) # double check

# fix the remaining spelling differences across years so each metro is one string
name_fixes = {"ethekweni": "ethekwini","ekhurhuleni": "ekurhuleni","non_metro": "non-metro"}
southafrica_df["metro_clean"] = southafrica_df["metro_clean"].replace(name_fixes)

# fix Indus variable (type of work ie. construction vs service) to standarize industry names
southafrica_df["indus_clean"] = southafrica_df["indus"].astype(str).str.replace(";", ",", regex=False)
##
# create list of hosting metros
southafrica_hosts = ["cape town", "nelson mandela metro", "ethekwini", "tshwane", "johannesburg"]

# use lamda function to sort into hosting metro and nonhost
southafrica_df["host_group"] = southafrica_df["metro_clean"].apply(
    lambda x: "Metro Host" if x in southafrica_hosts else "Non-Host")

print(southafrica_df["metro_clean"].unique())
print(southafrica_df.groupby(["year", "host_group"]).size())

['non-metro' 'cape town' 'nelson mandela metro' 'ethekweni' 'tshwane'
 'ekhurhuleni' 'johannesburg' 'non_metro' 'ethekwini' 'ekurhuleni']
['non-metro' 'cape town' 'nelson mandela metro' 'ethekwini' 'tshwane'
 'ekurhuleni' 'johannesburg']
year  host_group
2008  Metro Host    22347
      Non-Host      70715
2010  Metro Host    19039
      Non-Host      64318
2012  Metro Host    20765
      Non-Host      64347
dtype: int64


### Brazil

In [21]:
# 2014 tournament used 12 host cities; 8 appear in PNAD's 20 metro regions, so the
# remaining metros are not a clean control group. Restrict to 3 and 3,
# matched on size to the best i could (see README methods section)
brazil_hosts = ["Salvador (BA)", "Recife (PE)", "Fortaleza (CE)"] # hosting metros to use
brazil_nonhosts = ["Goiânia (GO)", "Belém (PA)", "Grande Vitória (ES)"] # non hosting to use
brazil_keep = brazil_hosts + brazil_nonhosts # combine the two lists as the places to keep

# function to speed up process since doing this across multiple files
def flag_hosts(brazil_df):
    brazil_df = brazil_df[brazil_df["metro"].isin(brazil_keep)].copy() #independent thing to mess with via copy
    brazil_df["host_group"] = brazil_df["metro"].apply(
        lambda x: "Metro Host" if x in brazil_hosts else "Non-Host") # lamba function, same logic as above with South Africa
    return brazil_df

# run the functions on the 4 datasets
brazil_contrib = flag_hosts(brazil_contrib)
brazil_income = flag_hosts(brazil_income)
brazil_hours = flag_hosts(brazil_hours)
brazil_lstatus = flag_hosts(brazil_lstatus)

# calculate informality
# south africa has a variable in the dataset for this, but Brazil doesn't
# use formula informal = non contribute to social security / total employed (since noncontribute is in formal/not registered with govt)

# for loop to iterate over appropriate rules
for year in [2012, 2014, 2016]:
    brazil_contrib[f"informal_{year}"] = (brazil_contrib[f"noncontrib_{year}"] / brazil_contrib[f"emp_{year}"] * 100)

# below should print to confirm loop worked:
#             informal_2012  informal_2014  informal_2016
# host_group                                             
# Metro Host      37.939606      34.988867      35.982475
# Non-Host        36.511927      34.501480      36.162344
print(brazil_contrib.groupby("host_group")[["informal_2012", "informal_2014", "informal_2016"]].mean())

print(brazil_contrib[["metro", "host_group"]]) # should list 6 cities, 3 host + 3 nonhost

            informal_2012  informal_2014  informal_2016
host_group                                             
Metro Host      37.939606      34.988867      35.982475
Non-Host        36.511927      34.501480      36.162344
                  metro  host_group
1            Belém (PA)    Non-Host
4        Fortaleza (CE)  Metro Host
7           Recife (PE)  Metro Host
10        Salvador (BA)  Metro Host
12  Grande Vitória (ES)    Non-Host
19         Goiânia (GO)    Non-Host


### Save files for 02_analysis

In [22]:
# save updated files back to data folder, all labeled with analysis to indicate ready for next step
southafrica_df.to_stata(data + "southafrica_analysis.dta", write_index=False)
brazil_contrib.to_csv(data + "brazil_socialsecurity_analysis.csv", index=False)
brazil_income.to_csv(data + "brazil_income_analysis.csv", index=False)
brazil_hours.to_csv(data + "brazil_hours_analysis.csv", index=False)
print("Saved files")

Saved files
